# 04 Feature Engineering

## Introduction

Feature engineering is one of the most important stages in the machine learning workflow. While raw data often contains useful information, it is rarely in the optimal format for predictive modeling. Transforming, encoding, and selecting features enables machine learning algorithms to identify meaningful relationships more effectively.

In this notebook, the cleaned weather dataset produced during preprocessing is transformed into a model-ready dataset. Several reusable feature engineering techniques are implemented, including datetime feature extraction, categorical encoding, outlier handling, feature scaling, feature selection, and construction of an end-to-end preprocessing pipeline.

All feature engineering operations are implemented using reusable Python modules developed specifically for this project, ensuring consistency, maintainability, and reproducibility throughout the machine learning workflow.

# 2. Objectives

The objectives of this notebook are to:

- Load the cleaned weather dataset.
- Apply reusable feature engineering utilities.
- Extract meaningful datetime features.
- Encode categorical variables for machine learning.
- Reduce the influence of extreme outliers.
- Standardize numerical variables through feature scaling.
- Select informative features for predictive modeling.
- Demonstrate reusable utility functions.
- Build an end-to-end feature engineering pipeline.
- Produce a model-ready dataset for subsequent machine learning experiments.

# 3. Configure Project Path

## Objective

To ensure the notebook can reliably import the project's custom Python modules regardless of where it is executed, the project root directory is added to Python's module search path (`sys.path`).

This makes the notebook self-contained and reproducible, allowing all modules within the `src` package to be imported consistently without manually changing the working directory.

In [58]:


from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root: {PROJECT_ROOT}")

Project Root: c:\Users\caleb\Documents\Projects\Weather-Forecasting-PM-Accelerator


# 4. Import Libraries

## Objective

This section imports the Python libraries and custom project modules required throughout the feature engineering workflow.

The notebook uses standard scientific computing libraries, including NumPy and pandas, together with reusable modules developed during earlier stages of the project. Centralizing all imports at the beginning of the notebook improves readability, maintainability, and reproducibility.

In [59]:


import numpy as np
import pandas as pd

from src.data_loader import load_weather_dataset
from src.preprocessing import preprocess_dataset

from src.feature_engineering import (
    extract_datetime_features,
    encode_categorical_features,
    clip_outliers_iqr,
    scale_features,
    select_numeric_features,
    remove_high_correlation,
    split_features_target,
    split_dataset,
    preprocess_pipeline,
)

# 5. Load and Preprocess the Dataset

## Objective

Before feature engineering techniques can be applied, the raw weather dataset must be loaded and preprocessed. This step ensures that the data is clean, consistent, and ready for feature transformation.

The preprocessing pipeline developed in Notebook 02 is reused to promote modularity and reproducibility across the project. By using the same preprocessing functions, every notebook begins with an identical cleaned dataset, reducing inconsistencies during model development.

In [60]:


df = load_weather_dataset()

df = preprocess_dataset(df)

2026-07-09 13:12:23 | INFO | src.data_loader | Loading weather dataset...


2026-07-09 13:12:24 | INFO | src.data_loader | Dataset loaded successfully (151827 rows, 41 columns).
2026-07-09 13:12:24 | INFO | src.preprocessing | ============================================================
2026-07-09 13:12:24 | INFO | src.preprocessing | Starting preprocessing pipeline...
2026-07-09 13:12:24 | INFO | src.preprocessing | ============================================================
2026-07-09 13:12:24 | INFO | src.preprocessing | Converting last_updated to datetime...
2026-07-09 13:12:24 | INFO | src.preprocessing | 0 duplicate rows removed.
2026-07-09 13:12:24 | INFO | src.preprocessing | Sorting dataset by last_updated...
2026-07-09 13:12:24 | INFO | src.preprocessing | Preprocessing complete.


In [61]:
print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (151827, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,United States of America,Washington Park,46.60,-120.49,America/Los_Angeles,1715849100,2024-05-16 01:45:00,16.1,61.0,Clear,...,6.3,7.1,1,1,05:26 AM,08:31 PM,01:36 PM,02:52 AM,Waxing Gibbous,55
1,Costa Rica,San Juan,9.97,-84.08,America/Costa_Rica,1715849100,2024-05-16 02:45:00,21.0,69.8,Fog,...,21.7,23.3,2,2,05:15 AM,05:51 PM,12:42 PM,12:37 AM,Waxing Gibbous,55
2,Mexico,Mexico City,19.43,-99.13,America/Mexico_City,1715849100,2024-05-16 02:45:00,20.8,69.4,Clear,...,35.1,48.0,2,3,06:01 AM,07:05 PM,01:37 PM,01:49 AM,Waxing Gibbous,55
3,El Salvador,San Salvador,13.71,-89.20,America/El_Salvador,1715849100,2024-05-16 02:45:00,26.0,78.8,Moderate or heavy rain with thunder,...,20.4,28.1,2,2,05:30 AM,06:16 PM,01:00 PM,01:02 AM,Waxing Gibbous,55
4,Guatemala,Guatemala City,14.62,-90.53,America/Guatemala,1715849100,2024-05-16 02:45:00,20.0,68.0,Mist,...,132.0,178.1,4,10,05:34 AM,06:23 PM,01:05 PM,01:09 AM,Waxing Gibbous,55


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151827 entries, 0 to 151826
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   country                       151827 non-null  object        
 1   location_name                 151827 non-null  object        
 2   latitude                      151827 non-null  float64       
 3   longitude                     151827 non-null  float64       
 4   timezone                      151827 non-null  object        
 5   last_updated_epoch            151827 non-null  int64         
 6   last_updated                  151827 non-null  datetime64[ns]
 7   temperature_celsius           151827 non-null  float64       
 8   temperature_fahrenheit        151827 non-null  float64       
 9   condition_text                151827 non-null  object        
 10  wind_mph                      151827 non-null  float64       
 11  wind_kph     

In [63]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
latitude,151827.0,19.238971,-41.3,4.0503,17.25,40.4,65.3,24.400579
longitude,151827.0,21.88928,-175.2,-6.8361,23.2361,49.8822,179.22,65.779044
last_updated_epoch,151827.0,1749640174.522318,1715849100.0,1732791600.0,1749632400.0,1766474100.0,1783404000.0,19507217.150295
last_updated,151827,2025-06-11 13:19:10.544369664,2024-05-16 01:45:00,2024-11-28 13:00:00,2025-06-11 12:00:00,2025-12-23 08:15:00,2026-07-07 19:00:00,NaN
temperature_celsius,151827.0,21.338242,-29.8,16.0,23.7,27.9,79.3,9.507643
temperature_fahrenheit,151827.0,70.410613,-21.6,60.9,74.6,82.2,174.7,17.113616
wind_mph,151827.0,7.941632,2.2,3.8,6.7,11.0,1841.2,7.012981
wind_kph,151827.0,12.784579,3.6,6.1,10.8,17.6,2963.2,11.282857
wind_degree,151827.0,169.518636,1.0,81.0,162.0,256.0,360.0,103.525664
pressure_mb,151827.0,1014.063329,947.0,1010.0,1014.0,1018.0,3006.0,9.985458


## Discussion

The dataset was successfully loaded and preprocessed using the reusable preprocessing pipeline developed earlier in the project. The preprocessing stage standardized datetime values, removed duplicate records, and ensured that the dataset was suitable for feature engineering.

Using a common preprocessing workflow improves reproducibility and guarantees that subsequent feature transformations are performed on a consistent dataset.

# 6. Datetime Feature Engineering

## Objective

Temporal information plays an important role in weather forecasting because atmospheric conditions often exhibit seasonal, monthly, weekly, and hourly patterns.

Instead of using the original timestamp directly, meaningful calendar features are extracted from the `last_updated` column. These engineered features enable machine learning algorithms to capture recurring temporal trends that may improve predictive performance.

In [64]:

df_datetime = extract_datetime_features(df)

In [65]:
datetime_columns = [
    "year",
    "month",
    "day",
    "hour",
    "day_of_week",
]

df_datetime[datetime_columns].head()

,year,month,day,hour,day_of_week
0,2024,5,16,1,3
1,2024,5,16,2,3
2,2024,5,16,2,3
3,2024,5,16,2,3
4,2024,5,16,2,3


In [66]:
print("Datetime Features Created")

for column in datetime_columns:
    print(f"✓ {column}")

Datetime Features Created
✓ year
✓ month
✓ day
✓ hour
✓ day_of_week


In [67]:
df_datetime[datetime_columns].describe()

,year,month,day,hour,day_of_week
count,151827.000000,151827.000000,151827.000000,151827.000000,151827.000000
mean,2024.945919,6.487562,15.758515,10.686788,3.005980
std,0.727174,3.336992,8.836837,4.769623,1.993869
min,2024.000000,1.000000,1.000000,0.000000,0.000000
25%,2024.000000,4.000000,8.000000,8.000000,1.000000
50%,2025.000000,6.000000,16.000000,11.000000,3.000000
75%,2025.000000,9.000000,23.000000,14.000000,5.000000
max,2026.000000,12.000000,31.000000,23.000000,6.000000


## Discussion

The original timestamp was successfully transformed into multiple calendar-based variables. These features provide machine learning models with additional contextual information regarding seasonal, monthly, daily, and hourly weather patterns.

Feature extraction reduces the complexity of working with raw datetime values while preserving valuable temporal information that can improve forecasting performance.

# 7. Categorical Feature Encoding

## Objective

Most machine learning algorithms require numerical input and therefore cannot directly process categorical variables such as weather conditions, countries, wind directions, or moon phases.

This section applies One-Hot Encoding to convert categorical variables into binary numerical features while preserving all category information.

In [68]:

categorical_columns = [
    "country",
    "condition_text",
    "wind_direction",
    "moon_phase",
]

df_encoded = encode_categorical_features(
    df_datetime,
    columns=categorical_columns,
    method="onehot",
)

In [69]:
print("Original Dataset Shape :", df_datetime.shape)

print("Encoded Dataset Shape  :", df_encoded.shape)

Original Dataset Shape : (151827, 49)
Encoded Dataset Shape  : (151827, 337)


In [70]:
encoded_columns = [
    column
    for column in df_encoded.columns
    if any(
        feature in column
        for feature in categorical_columns
    )
]

print("New Encoded Features:", len(encoded_columns))

New Encoded Features: 292


In [71]:
encoded_columns[:20]

['country_Afghanistan',
 'country_Albania',
 'country_Algeria',
 'country_Andorra',
 'country_Angola',
 'country_Antigua and Barbuda',
 'country_Argentina',
 'country_Armenia',
 'country_Australia',
 'country_Austria',
 'country_Azerbaijan',
 'country_Bahamas',
 'country_Bahrain',
 'country_Bangladesh',
 'country_Barbados',
 'country_Belarus',
 'country_Belgium',
 'country_Belize',
 'country_Benin',
 'country_Bhutan']

## Discussion

One-Hot Encoding successfully transformed all categorical variables into binary numerical features. Although this process increased the dimensionality of the dataset, it preserved all categorical information without introducing artificial ordinal relationships between categories.

This representation is particularly suitable for tree-based machine learning algorithms and ensures compatibility with most supervised learning models.

# 8. Outlier Handling

## Objective

Extreme observations can negatively affect statistical analysis and machine learning performance. Instead of removing observations entirely, this project applies Interquartile Range (IQR) clipping to reduce the influence of unusually large or small values while preserving the overall dataset size.

This approach is especially appropriate for weather datasets where unusual observations may represent legitimate but infrequent environmental conditions.

In [72]:

numeric_columns = [
    "temperature_celsius",
    "humidity",
    "wind_kph",
    "pressure_mb",
    "visibility_km",
    "uv_index",
    "precip_mm",
]

df_outliers = clip_outliers_iqr(
    df_encoded,
    columns=numeric_columns,
)

In [73]:
df_outliers[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
temperature_celsius,151827.0,21.434381,9.207244,-1.85,16.0,23.7,27.90,45.75
humidity,151827.0,66.925586,23.654361,2.00,51.0,72.0,86.00,100.00
wind_kph,151827.0,12.673989,7.991895,3.60,6.1,10.8,17.60,34.85
pressure_mb,151827.0,1014.053317,6.512512,998.00,1010.0,1014.0,1018.00,1030.00
visibility_km,151827.0,10.000000,0.000000,10.00,10.0,10.0,10.00,10.00
uv_index,151827.0,3.203367,3.513022,0.00,0.0,1.7,6.00,15.00
precip_mm,151827.0,0.012952,0.020575,0.00,0.0,0.0,0.02,0.05


In [74]:
comparison = pd.DataFrame({
    "Before": df_encoded[numeric_columns].max(),
    "After": df_outliers[numeric_columns].max(),
})

comparison

,Before,After
temperature_celsius,79.30,45.75
humidity,100.00,100.00
wind_kph,2963.20,34.85
pressure_mb,3006.00,1030.00
visibility_km,32.00,10.00
uv_index,16.30,15.00
precip_mm,42.24,0.05


## Discussion

Outlier clipping successfully reduced the influence of extreme observations while retaining all available records. This approach improves the robustness of subsequent machine learning models by limiting the effect of anomalous measurements, such as unrealistic wind speeds and atmospheric pressure values identified during exploratory data analysis.

Unlike outlier removal, clipping preserves valuable observations while preventing extreme values from dominating model training.

# 9. Feature Scaling

## Objective

Many machine learning algorithms are sensitive to differences in feature magnitude. Variables measured on larger scales can dominate the learning process, leading to biased model performance.

To address this issue, numerical features are standardized using the Standard Scaler. Standardization transforms each feature to have an approximately zero mean and unit standard deviation while preserving the underlying distribution of the data.

This preprocessing step is particularly beneficial for algorithms such as Linear Regression, Logistic Regression, Support Vector Machines, K-Nearest Neighbors, Neural Networks, and Principal Component Analysis.

In [75]:

numeric_columns = [
    "temperature_celsius",
    "humidity",
    "wind_kph",
    "pressure_mb",
    "visibility_km",
    "uv_index",
    "precip_mm",
]

df_scaled, scaler = scale_features(
    df=df_outliers,
    columns=numeric_columns,
    method="standard",
)

In [76]:
df_scaled.head()

,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,wind_mph,wind_kph,...,wind_direction_WNW,wind_direction_WSW,moon_phase_First Quarter,moon_phase_Full Moon,moon_phase_Last Quarter,moon_phase_New Moon,moon_phase_Waning Crescent,moon_phase_Waning Gibbous,moon_phase_Waxing Crescent,moon_phase_Waxing Gibbous
0,Washington Park,46.60,-120.49,America/Los_Angeles,1715849100,2024-05-16 01:45:00,-0.579370,61.0,4.3,-0.734996,...,0,0,0,0,0,0,0,0,0,1
1,San Juan,9.97,-84.08,America/Costa_Rica,1715849100,2024-05-16 02:45:00,-0.047178,69.8,2.2,-1.135403,...,0,0,0,0,0,0,0,0,0,1
2,Mexico City,19.43,-99.13,America/Mexico_City,1715849100,2024-05-16 02:45:00,-0.068900,69.4,6.7,-0.234487,...,0,0,0,0,0,0,0,0,0,1
3,San Salvador,13.71,-89.20,America/El_Salvador,1715849100,2024-05-16 02:45:00,0.495874,78.8,2.2,-1.135403,...,0,0,0,0,0,0,0,0,0,1
4,Guatemala City,14.62,-90.53,America/Guatemala,1715849100,2024-05-16 02:45:00,-0.155789,68.0,13.6,1.166937,...,0,0,0,0,0,0,0,0,0,1


In [77]:
df_scaled[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
temperature_celsius,151827.0,-1.497584e-18,1.000003,-2.528928,-0.590231,0.246070,0.702234,2.640931
humidity,151827.0,-2.119081e-16,1.000003,-2.744771,-0.673264,0.214524,0.806383,1.398242
wind_kph,151827.0,-3.444443e-17,1.000003,-1.135403,-0.822585,-0.234487,0.616378,2.774822
pressure_mb,151827.0,5.072317e-15,1.000003,-2.465004,-0.622391,-0.008187,0.606017,2.448631
visibility_km,151827.0,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
uv_index,151827.0,8.985504e-18,1.000003,-0.911858,-0.911858,-0.427943,0.796079,3.357984
precip_mm,151827.0,1.437681e-16,1.000003,-0.629517,-0.629517,-0.629517,0.342537,1.800616


In [78]:
print("Feature Means")

display(
    df_scaled[numeric_columns]
    .mean()
    .round(4)
)

Feature Means


temperature_celsius   -0.0
humidity              -0.0
wind_kph              -0.0
pressure_mb            0.0
visibility_km          0.0
uv_index               0.0
precip_mm              0.0
dtype: float64

In [79]:
print("Feature Standard Deviations")

display(
    df_scaled[numeric_columns]
    .std()
    .round(4)
)

Feature Standard Deviations


temperature_celsius    1.0
humidity               1.0
wind_kph               1.0
pressure_mb            1.0
visibility_km          0.0
uv_index               1.0
precip_mm              1.0
dtype: float64

In [80]:
print(type(scaler))

<class 'sklearn.preprocessing._data.StandardScaler'>


## Discussion

The selected numerical variables were successfully standardized using the Standard Scaler. After scaling, each feature has an approximately zero mean and unit standard deviation, making all variables comparable despite their original units of measurement.

Returning the fitted scaler alongside the transformed dataset is an important design decision. The same scaler can later be applied to validation, testing, or production data, ensuring that new observations are transformed consistently with the training data.

Standardization improves numerical stability and optimization during model training and is especially beneficial for machine learning algorithms that rely on distance calculations or gradient-based optimization.

The dataset has now undergone the primary feature engineering transformations, including datetime feature extraction, categorical encoding, outlier handling, and feature scaling.

The next step is **Feature Selection**, where redundant variables are identified and removed to reduce dimensionality, improve computational efficiency, and minimize multicollinearity before model development.

# 10. Feature Selection

## Objective

After feature engineering, the dataset contains a large number of variables, including engineered features and encoded categorical variables. Not all of these features contribute equally to predictive performance, and some may contain redundant information.

The objective of feature selection is to identify informative variables while reducing redundancy and multicollinearity. Removing highly correlated features simplifies the dataset, improves computational efficiency, and may enhance the performance and interpretability of machine learning models.

This section demonstrates the reusable feature selection utilities developed for this project.

In [81]:

numeric_df = select_numeric_features(df_scaled)

print("Numeric Dataset Shape:")
print(numeric_df.shape)

Numeric Dataset Shape:
(151827, 330)


In [82]:
numeric_df.columns.tolist()

['latitude',
 'longitude',
 'last_updated_epoch',
 'temperature_celsius',
 'temperature_fahrenheit',
 'wind_mph',
 'wind_kph',
 'wind_degree',
 'pressure_mb',
 'pressure_in',
 'precip_mm',
 'precip_in',
 'humidity',
 'cloud',
 'feels_like_celsius',
 'feels_like_fahrenheit',
 'visibility_km',
 'visibility_miles',
 'uv_index',
 'gust_mph',
 'gust_kph',
 'air_quality_Carbon_Monoxide',
 'air_quality_Ozone',
 'air_quality_Nitrogen_dioxide',
 'air_quality_Sulphur_dioxide',
 'air_quality_PM2.5',
 'air_quality_PM10',
 'air_quality_us-epa-index',
 'air_quality_gb-defra-index',
 'moon_illumination',
 'year',
 'month',
 'day',
 'hour',
 'day_of_week',
 'week_of_year',
 'quarter',
 'is_weekend',
 'country_Afghanistan',
 'country_Albania',
 'country_Algeria',
 'country_Andorra',
 'country_Angola',
 'country_Antigua and Barbuda',
 'country_Argentina',
 'country_Armenia',
 'country_Australia',
 'country_Austria',
 'country_Azerbaijan',
 'country_Bahamas',
 'country_Bahrain',
 'country_Bangladesh',
 '

### Discussion

The dataset contains numerous numerical variables, including original weather measurements, engineered datetime variables, and binary features generated through One-Hot Encoding.

Selecting only numerical features ensures compatibility with correlation analysis and many machine learning algorithms that operate exclusively on numerical data.

### Removing Highly Correlated Features

Highly correlated variables often provide nearly identical information. Retaining both features may introduce unnecessary redundancy and increase model complexity.

The reusable feature selection function removes variables whose absolute Pearson correlation exceeds a specified threshold.

In [83]:
reduced_df = remove_high_correlation(
    numeric_df,
    threshold=0.90,
)

print("Before:", numeric_df.shape)

print("After :", reduced_df.shape)

Before: (151827, 330)
After : (151827, 321)


In [84]:
removed_features = set(numeric_df.columns) - set(reduced_df.columns)

print("Removed Features")

sorted(list(removed_features))

Removed Features


['air_quality_gb-defra-index',
 'feels_like_celsius',
 'feels_like_fahrenheit',
 'gust_kph',
 'gust_mph',
 'quarter',
 'temperature_fahrenheit',
 'week_of_year',
 'year']

### Discussion

Several highly correlated variables were successfully removed from the dataset. Most eliminated features represented duplicate measurements expressed in alternative units, such as Fahrenheit versus Celsius or miles versus kilometers.

Removing redundant variables reduces dimensionality without sacrificing meaningful information and helps mitigate multicollinearity during model development.

### Splitting Features and Target

Machine learning models require the predictor variables (features) and the response variable (target) to be separated before training.

For this project, the target variable is **temperature_celsius**, which will be predicted using the remaining weather measurements.

In [85]:
X, y = split_features_target(
    reduced_df,
    target="temperature_celsius",
)

In [86]:
print("Feature Matrix Shape")

print(X.shape)

Feature Matrix Shape
(151827, 320)


In [87]:
print("Target Shape")

print(y.shape)

Target Shape
(151827,)


In [88]:
X.head()

,latitude,longitude,last_updated_epoch,wind_mph,wind_kph,wind_degree,pressure_mb,pressure_in,precip_mm,precip_in,...,wind_direction_WNW,wind_direction_WSW,moon_phase_First Quarter,moon_phase_Full Moon,moon_phase_Last Quarter,moon_phase_New Moon,moon_phase_Waning Crescent,moon_phase_Waning Gibbous,moon_phase_Waxing Crescent,moon_phase_Waxing Gibbous
0,46.60,-120.49,1715849100,4.3,-0.734996,220,-0.315289,29.87,-0.629517,0.00,...,0,0,0,0,0,0,0,0,0,1
1,9.97,-84.08,1715849100,2.2,-1.135403,10,0.298915,30.01,1.800616,0.01,...,0,0,0,0,0,0,0,0,0,1
2,19.43,-99.13,1715849100,6.7,-0.234487,212,-0.161738,29.92,-0.629517,0.00,...,0,0,0,0,0,0,0,0,0,1
3,13.71,-89.20,1715849100,2.2,-1.135403,182,-0.622391,29.81,1.800616,0.01,...,0,0,0,0,0,0,0,0,0,1
4,14.62,-90.53,1715849100,13.6,1.166937,190,0.759569,30.09,1.800616,0.00,...,0,0,0,0,0,0,0,0,0,1


In [89]:
y.head()

0   -0.579370
1   -0.047178
2   -0.068900
3    0.495874
4   -0.155789
Name: temperature_celsius, dtype: float64

### Discussion

The dataset was successfully divided into predictor variables and the target variable. Separating features from the response variable is an essential prerequisite for supervised machine learning algorithms and prepares the dataset for train-test splitting in the next stage.

### Creating Training and Testing Sets

To evaluate predictive performance objectively, the dataset is divided into separate training and testing subsets.

The training set is used to fit machine learning models, while the testing set provides an unbiased evaluation using previously unseen observations.

In [90]:
X_train, X_test, y_train, y_test = split_dataset(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

In [91]:
print("Training Features :", X_train.shape)

print("Testing Features  :", X_test.shape)

print("Training Target   :", y_train.shape)

print("Testing Target    :", y_test.shape)

Training Features : (121461, 320)
Testing Features  : (30366, 320)
Training Target   : (121461,)
Testing Target    : (30366,)


In [92]:
X_train.head()

,latitude,longitude,last_updated_epoch,wind_mph,wind_kph,wind_degree,pressure_mb,pressure_in,precip_mm,precip_in,...,wind_direction_WNW,wind_direction_WSW,moon_phase_First Quarter,moon_phase_Full Moon,moon_phase_Last Quarter,moon_phase_New Moon,moon_phase_Waning Crescent,moon_phase_Waning Gibbous,moon_phase_Waxing Crescent,moon_phase_Waxing Gibbous
56701,-35.2833,149.2167,1741080600,10.7,0.578840,96,1.527324,30.24,-0.629517,0.0,...,0,0,0,0,0,0,0,0,1,0
21978,-17.8200,31.0400,1725710400,8.9,0.215971,41,0.298915,30.00,-0.629517,0.0,...,0,0,0,0,0,0,0,0,1,0
69077,48.1500,17.1167,1746607500,8.9,0.215971,17,0.145364,29.97,-0.629517,0.0,...,0,0,0,0,0,0,0,0,0,1
126890,13.1000,-59.6167,1772346600,20.8,2.605900,86,-0.161738,29.91,-0.629517,0.0,...,0,0,0,0,0,0,0,0,0,1
42870,6.8833,73.1000,1734950700,11.4,0.716480,336,-0.775942,29.80,-0.629517,0.0,...,0,0,0,0,1,0,0,0,0,0


### Discussion

The dataset was successfully divided into training and testing subsets using an 80:20 split.

This strategy provides sufficient observations for model training while preserving an independent testing set for evaluating model performance. Using a fixed random state ensures that the split is reproducible across multiple executions.

The feature engineering workflow is now complete. The dataset has been transformed, cleaned, encoded, standardized, and prepared for machine learning.

The following sections demonstrate the reusable utility functions developed for this project before integrating every preprocessing step into a single end-to-end feature engineering pipeline.

# 11. Utility Functions

## Objective

Reusable utility functions simplify common machine learning tasks such as splitting datasets into training and testing subsets and saving trained objects for later use.

This project includes helper functions for dataset splitting as well as saving and loading Python objects using Joblib. These utilities promote code reuse and improve reproducibility throughout the machine learning pipeline.

In [93]:


from joblib import dump

Path("models").mkdir(parents=True, exist_ok=True)

dump(scaler, "models/scaler.joblib")

print("Scaler saved successfully.")

Scaler saved successfully.


In [94]:
from pathlib import Path

Path("models/scaler.joblib").exists()

True

### Discussion

The fitted StandardScaler was successfully serialized and saved to disk using Joblib.

Saving preprocessing objects ensures that identical transformations can be applied to new observations during model deployment, maintaining consistency between training and inference.

In [95]:
from joblib import load as load_object

loaded_scaler = load_object("models/scaler.joblib")


In [96]:
print(type(loaded_scaler))

<class 'sklearn.preprocessing._data.StandardScaler'>


### Discussion

The previously saved scaler was successfully restored from disk. This demonstrates that preprocessing objects can be reused without retraining, which is essential for reproducible machine learning workflows and production deployment.

# 12. End-to-End Feature Engineering Pipeline

## Objective

Rather than performing each feature engineering step individually, a reusable preprocessing pipeline combines all transformations into a single function.

The pipeline automates the complete workflow, including datetime feature extraction, categorical encoding, outlier handling, feature scaling, and feature selection. This modular approach reduces code duplication, improves maintainability, and ensures that identical preprocessing is applied whenever new data is introduced.

In [97]:

pipeline_df, pipeline_scaler = preprocess_pipeline(
    df=df,
    categorical_columns=categorical_columns,
    numerical_columns=numeric_columns,
)

## Remove Target Leakage Features

Certain variables are directly derived from the prediction target (`temperature_celsius`) and therefore introduce target leakage into machine learning models. These variables are removed before saving the engineered dataset to ensure that model evaluation reflects realistic predictive performance.

In [98]:
leakage_columns = [
    "temperature_fahrenheit",
    "feels_like_celsius",
    "feels_like_fahrenheit",
]

pipeline_df = pipeline_df.drop(
    columns=leakage_columns
)

print(f"New Shape: {pipeline_df.shape}")

New Shape: (151827, 334)


In [99]:
pipeline_df.head()

,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,wind_mph,wind_kph,wind_degree,...,wind_direction_WNW,wind_direction_WSW,moon_phase_First Quarter,moon_phase_Full Moon,moon_phase_Last Quarter,moon_phase_New Moon,moon_phase_Waning Crescent,moon_phase_Waning Gibbous,moon_phase_Waxing Crescent,moon_phase_Waxing Gibbous
0,Washington Park,46.60,-120.49,America/Los_Angeles,1715849100,2024-05-16 01:45:00,-0.579370,4.3,-0.734996,220,...,0,0,0,0,0,0,0,0,0,1
1,San Juan,9.97,-84.08,America/Costa_Rica,1715849100,2024-05-16 02:45:00,-0.047178,2.2,-1.135403,10,...,0,0,0,0,0,0,0,0,0,1
2,Mexico City,19.43,-99.13,America/Mexico_City,1715849100,2024-05-16 02:45:00,-0.068900,6.7,-0.234487,212,...,0,0,0,0,0,0,0,0,0,1
3,San Salvador,13.71,-89.20,America/El_Salvador,1715849100,2024-05-16 02:45:00,0.495874,2.2,-1.135403,182,...,0,0,0,0,0,0,0,0,0,1
4,Guatemala City,14.62,-90.53,America/Guatemala,1715849100,2024-05-16 02:45:00,-0.155789,13.6,1.166937,190,...,0,0,0,0,0,0,0,0,0,1


In [100]:
print("Pipeline Dataset Shape")

print(pipeline_df.shape)

Pipeline Dataset Shape
(151827, 334)


In [101]:
pipeline_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151827 entries, 0 to 151826
Columns: 334 entries, location_name to moon_phase_Waxing Gibbous
dtypes: datetime64[ns](1), float64(21), int32(300), int64(6), object(6)
memory usage: 213.1+ MB


In [102]:
print(type(pipeline_scaler))

<class 'sklearn.preprocessing._data.StandardScaler'>


### Discussion

The reusable preprocessing pipeline successfully executed the complete feature engineering workflow. Datetime features were extracted, categorical variables were encoded, numerical outliers were clipped, and selected numerical features were standardized.

In addition to returning the transformed dataset, the pipeline also returned the fitted scaler. Retaining the scaler is important because the identical transformation can later be applied to unseen data during model evaluation or deployment, ensuring consistency between training and inference.

Encapsulating multiple preprocessing steps into a single reusable function improves maintainability, reduces code duplication, and supports reproducible machine learning workflows.

In [103]:
print("=" * 60)
print("FINAL FEATURE ENGINEERING DATASET")
print("=" * 60)

print(f"Rows    : {pipeline_df.shape[0]:,}")
print(f"Columns : {pipeline_df.shape[1]:,}")

pipeline_df.head()

FINAL FEATURE ENGINEERING DATASET
Rows    : 151,827
Columns : 334


,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,wind_mph,wind_kph,wind_degree,...,wind_direction_WNW,wind_direction_WSW,moon_phase_First Quarter,moon_phase_Full Moon,moon_phase_Last Quarter,moon_phase_New Moon,moon_phase_Waning Crescent,moon_phase_Waning Gibbous,moon_phase_Waxing Crescent,moon_phase_Waxing Gibbous
0,Washington Park,46.60,-120.49,America/Los_Angeles,1715849100,2024-05-16 01:45:00,-0.579370,4.3,-0.734996,220,...,0,0,0,0,0,0,0,0,0,1
1,San Juan,9.97,-84.08,America/Costa_Rica,1715849100,2024-05-16 02:45:00,-0.047178,2.2,-1.135403,10,...,0,0,0,0,0,0,0,0,0,1
2,Mexico City,19.43,-99.13,America/Mexico_City,1715849100,2024-05-16 02:45:00,-0.068900,6.7,-0.234487,212,...,0,0,0,0,0,0,0,0,0,1
3,San Salvador,13.71,-89.20,America/El_Salvador,1715849100,2024-05-16 02:45:00,0.495874,2.2,-1.135403,182,...,0,0,0,0,0,0,0,0,0,1
4,Guatemala City,14.62,-90.53,America/Guatemala,1715849100,2024-05-16 02:45:00,-0.155789,13.6,1.166937,190,...,0,0,0,0,0,0,0,0,0,1


In [104]:
print("Total Missing Values:")

print(
    pipeline_df
    .isna()
    .sum()
    .sum()
)

Total Missing Values:
0


In [105]:
print("Duplicate Rows:")

print(
    pipeline_df
    .duplicated()
    .sum()
)

Duplicate Rows:


0


In [106]:
pipeline_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151827 entries, 0 to 151826
Columns: 334 entries, location_name to moon_phase_Waxing Gibbous
dtypes: datetime64[ns](1), float64(21), int32(300), int64(6), object(6)
memory usage: 213.1+ MB


In [107]:
memory = (
    pipeline_df.memory_usage(deep=True)
    .sum()
    / 1024**2
)

print(f"Memory Usage: {memory:.2f} MB")

Memory Usage: 256.59 MB


In [108]:
pipeline_df.to_csv(
    "../data/processed/weather_engineered.csv",
    index=False,
)

print("Feature-engineered dataset saved successfully.")

Feature-engineered dataset saved successfully.


# 13. Summary

## Summary

This notebook transformed the cleaned weather dataset into a machine learning-ready dataset through a structured feature engineering workflow.

The completed workflow included:

- Loading the cleaned weather dataset.
- Applying the reusable preprocessing pipeline.
- Extracting informative datetime features.
- Encoding categorical variables using One-Hot Encoding.
- Reducing the influence of extreme observations through IQR clipping.
- Standardizing numerical variables using Standard Scaling.
- Selecting numerical variables for machine learning.
- Removing highly correlated features.
- Separating predictor variables from the target variable.
- Splitting the dataset into training and testing subsets.
- Demonstrating reusable utility functions for model persistence.
- Executing the complete feature engineering pipeline.
- Saving the final engineered dataset for downstream model development.

The resulting dataset is fully prepared for supervised machine learning and can now be used to train predictive weather forecasting models.

# 14. Conclusion

Feature engineering represents one of the most important stages of the machine learning lifecycle because the quality of engineered features directly influences predictive performance.

Throughout this notebook, raw weather observations were transformed into a structured numerical dataset suitable for machine learning. Temporal information was extracted from timestamps, categorical variables were encoded, numerical variables were standardized, redundant features were reduced, and reusable preprocessing utilities were developed.

By organizing these transformations into modular Python functions and an end-to-end preprocessing pipeline, the project now follows software engineering best practices that emphasize maintainability, reproducibility, and scalability.

The completed feature engineering pipeline provides a reliable foundation for the next stage of the project: machine learning model development and evaluation.